# 05 Prefill Estimator

This notebook introduces a simple simulation for prefill work and TTFT intuition.

Prefill means the model processes all input prompt tokens before it can generate the first output token.

TTFT means **Time To First Token**.

## Simulation Formula

This step is only a simulation, not a real benchmark.

We use:
- `base_ttft_ms = 100`
- `prefill_time_per_1k_tokens_ms = 40`

Formula:

`estimated_ttft_ms = base_ttft_ms + (prompt_tokens / 1000) * prefill_time_per_1k_tokens_ms`

Main intuition:
- more prompt tokens -> more prefill work -> higher TTFT

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from prefill_estimator import (
    CONTEXT_REPORT_PATH,
    FIGURES_DIR,
    PREFILL_REPORT_PATH,
    PROMPT_PACKING_REPORT_PATH,
    build_prefill_report,
    build_prefill_summary,
    create_prefill_graphs,
    load_previous_reports,
    save_prefill_report,
)

print("Project root:", PROJECT_ROOT)
print("Packing report path:", PROMPT_PACKING_REPORT_PATH)
print("Context report path:", CONTEXT_REPORT_PATH)
print("Prefill report path:", PREFILL_REPORT_PATH)
print("Figures directory:", FIGURES_DIR)

Project root: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference
Packing report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/prompt_packing_report.csv
Context report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/context_window_report.csv
Prefill report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/prefill_estimation_report.csv
Figures directory: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/figures


## Load Previous Report

If the Step 4 prompt packing report exists, we use `packed_prompt_tokens` as the input token count.
If not, we fall back to `prompt_token_count` from the Step 3 context report.

In [2]:
source_df, source_name = load_previous_reports()
source_name, source_df.head()

('packing',
                        tokenizer_name  context_window  reserved_output_tokens  \
 0  mistralai/Mistral-7B-Instruct-v0.2            4096                     512   
 1  TinyLlama/TinyLlama-1.1B-Chat-v1.0            4096                     512   
 2          Qwen/Qwen2.5-1.5B-Instruct            4096                     512   
 3          Qwen/Qwen2.5-1.5B-Instruct            8192                     512   
 4          Qwen/Qwen2.5-1.5B-Instruct           32768                     512   
 
    available_input_budget  total_original_prompt_tokens  packed_prompt_tokens  \
 0                    3584                          3836                  2621   
 1                    3584                          3877                  2663   
 2                    3584                          3371                  3371   
 3                    7680                          3371                  3371   
 4                   32256                          3371                  3371   
 


## Build the Prefill Estimation Report

For each row, we calculate:
- prompt tokens
- prefill load category
- estimated TTFT in milliseconds
- estimated TTFT in seconds
- a short interpretation string

In [3]:
df = build_prefill_report(source_df, source_name)
df = save_prefill_report(df, PREFILL_REPORT_PATH)
df.head(10)

,scenario_name,tokenizer_name,context_window,reserved_output_tokens,prompt_tokens,prefill_load,base_ttft_ms,prefill_time_per_1k_tokens_ms,estimated_ttft_ms,estimated_ttft_seconds,interpretation
7,packed_prompt_8192,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,512,3877,medium,100,40,255.08,0.2551,"Using packed prompt tokens=3877, this scenario..."
8,packed_prompt_32768,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32768,512,3877,medium,100,40,255.08,0.2551,"Using packed prompt tokens=3877, this scenario..."
5,packed_prompt_8192,mistralai/Mistral-7B-Instruct-v0.2,8192,512,3836,medium,100,40,253.44,0.2534,"Using packed prompt tokens=3836, this scenario..."
6,packed_prompt_32768,mistralai/Mistral-7B-Instruct-v0.2,32768,512,3836,medium,100,40,253.44,0.2534,"Using packed prompt tokens=3836, this scenario..."
2,packed_prompt_4096,Qwen/Qwen2.5-1.5B-Instruct,4096,512,3371,medium,100,40,234.84,0.2348,"Using packed prompt tokens=3371, this scenario..."
3,packed_prompt_8192,Qwen/Qwen2.5-1.5B-Instruct,8192,512,3371,medium,100,40,234.84,0.2348,"Using packed prompt tokens=3371, this scenario..."
4,packed_prompt_32768,Qwen/Qwen2.5-1.5B-Instruct,32768,512,3371,medium,100,40,234.84,0.2348,"Using packed prompt tokens=3371, this scenario..."
1,packed_prompt_4096,TinyLlama/TinyLlama-1.1B-Chat-v1.0,4096,512,2663,medium,100,40,206.52,0.2065,"Using packed prompt tokens=2663, this scenario..."
0,packed_prompt_4096,mistralai/Mistral-7B-Instruct-v0.2,4096,512,2621,medium,100,40,204.84,0.2048,"Using packed prompt tokens=2621, this scenario..."


## Full Report Sorted by Estimated TTFT

The largest prompts appear at the top because they require the most simulated prefill work.

In [4]:
df

,scenario_name,tokenizer_name,context_window,reserved_output_tokens,prompt_tokens,prefill_load,base_ttft_ms,prefill_time_per_1k_tokens_ms,estimated_ttft_ms,estimated_ttft_seconds,interpretation
7,packed_prompt_8192,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,512,3877,medium,100,40,255.08,0.2551,"Using packed prompt tokens=3877, this scenario..."
8,packed_prompt_32768,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32768,512,3877,medium,100,40,255.08,0.2551,"Using packed prompt tokens=3877, this scenario..."
5,packed_prompt_8192,mistralai/Mistral-7B-Instruct-v0.2,8192,512,3836,medium,100,40,253.44,0.2534,"Using packed prompt tokens=3836, this scenario..."
6,packed_prompt_32768,mistralai/Mistral-7B-Instruct-v0.2,32768,512,3836,medium,100,40,253.44,0.2534,"Using packed prompt tokens=3836, this scenario..."
2,packed_prompt_4096,Qwen/Qwen2.5-1.5B-Instruct,4096,512,3371,medium,100,40,234.84,0.2348,"Using packed prompt tokens=3371, this scenario..."
3,packed_prompt_8192,Qwen/Qwen2.5-1.5B-Instruct,8192,512,3371,medium,100,40,234.84,0.2348,"Using packed prompt tokens=3371, this scenario..."
4,packed_prompt_32768,Qwen/Qwen2.5-1.5B-Instruct,32768,512,3371,medium,100,40,234.84,0.2348,"Using packed prompt tokens=3371, this scenario..."
1,packed_prompt_4096,TinyLlama/TinyLlama-1.1B-Chat-v1.0,4096,512,2663,medium,100,40,206.52,0.2065,"Using packed prompt tokens=2663, this scenario..."
0,packed_prompt_4096,mistralai/Mistral-7B-Instruct-v0.2,4096,512,2621,medium,100,40,204.84,0.2048,"Using packed prompt tokens=2621, this scenario..."


## Summary Findings

These summary values answer the main Step 5 questions:
- which scenario has the highest prefill load
- which tokenizer/context combination has the largest prompt size
- why long RAG prompts increase TTFT
- why long chat history increases TTFT

In [5]:
summary = build_prefill_summary(df)
summary

{'highest_prefill_load_scenario': 'packed_prompt_8192',
 'highest_prefill_load_tokenizer': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'highest_input_size_context_window': 8192,
 'why_long_rag_increases_ttft': 'Long RAG prompts add many input tokens, so the model must process more tokens before the first output token can arrive.',
 'why_long_chat_history_increases_ttft': 'Long chat history increases prompt tokens, which increases prefill work and pushes TTFT upward.',
 'average_estimated_ttft_by_context_window': {4096: 215.4,
  8192: 247.79,
  32768: 247.79}}

## Create Graphs

These charts help visualize how prompt size changes simulated prefill work and estimated TTFT.

In [ ]:
create_prefill_graphs(df, FIGURES_DIR)
sorted(path.name for path in FIGURES_DIR.glob('*.png'))

## What the Graphs Show

1. **Prefill load by prompt**
   This compares which scenarios create the heaviest prefill burden.

2. **Estimated TTFT by prompt**
   This shows the simulated delay before the first token arrives.

3. **Prompt tokens vs estimated TTFT**
   This scatter plot shows the direct relationship between prompt size and TTFT.

4. **Average estimated TTFT by context window**
   This shows how the scenarios grouped by context window compare in simulated prefill cost.

## Why This Is Only a Simulation

This notebook does **not** measure real TTFT.

A real benchmark would need:
- an actual loaded model
- a real inference request
- a timestamp before the request starts
- a timestamp when the first token arrives
- streaming response support
- repeated runs
- average, p50, and p95 TTFT measurements

## Step 5 Takeaway

More prompt tokens generally mean more prefill work. Even before generation begins, long RAG prompts, long chat history, and large tool outputs can push TTFT higher.